# Preview Copy-Paste Scene Kopi 300 g

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ediprin/coffee-bean-detection/blob/agent/add-vadcp-pipeline/notebooks/SNI_300g_CopyPaste_Preview_Colab.ipynb)

Notebook ini hanya membuat **4 preview A1/A2** dengan 220–300 biji per gambar. Rentang ini adalah pilot visual, bukan konversi pasti 300 g; jumlah final harus dikalibrasi dari foto 300 g nyata. Notebook tidak menyalin seluruh dataset dan **tidak menjalankan training**. A1 adalah copy-paste biasa; A2 menambahkan oklusi ringan yang terukur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if not REPO.is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
import coffee_detector
print('IMPORT BERHASIL:', coffee_detector.__file__)

## Atur dua lokasi input

Notebook mencari dataset YOLO yang sudah ada di `/content`. Jika belum ada, notebook meminta API key Roboflow secara tersembunyi lalu mengunduh `adrians/coffee-defect-1eny4` versi 11. `OBJECT_LIBRARY` diekstrak dari **split train saja**.

In [ ]:
from getpass import getpass
from coffee_detector.dataset import discover_layout

EXPECTED_ROOT = Path('/content/coffee-defect-roboflow')
OBJECT_LIBRARY = Path('/content/coffee-object-library')
OUTPUT_ROOT = Path('/content/drive/MyDrive/coffee-bean-detection/sni-300g-preview')

def find_yolo_dataset():
    candidates = [EXPECTED_ROOT]
    candidates += [p.parent for p in Path('/content').glob('*/data.yaml')]
    candidates += [p.parent for p in Path('/content').glob('*/*/data.yaml') if 'drive' not in p.parts]
    unique = []
    for candidate in candidates:
        if candidate in unique or candidate == REPO:
            continue
        unique.append(candidate)
        try:
            layout = discover_layout(candidate)
            if {'train', 'val', 'test'} <= set(layout.splits):
                return candidate
        except (FileNotFoundError, ValueError):
            pass
    return None

REAL_DATA_ROOT = find_yolo_dataset()
if REAL_DATA_ROOT is None:
    print('Dataset belum ada di runtime. Download Roboflow dimulai.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'], check=True)
    from roboflow import Roboflow
    api_key = getpass('Masukkan Roboflow API key: ')
    assert api_key.strip(), 'API key tidak boleh kosong'
    rf = Roboflow(api_key=api_key.strip())
    version = rf.workspace('adrians').project('coffee-defect-1eny4').version(11)
    version.download('yolov8', location=str(EXPECTED_ROOT), overwrite=True)
    del api_key
    REAL_DATA_ROOT = find_yolo_dataset()

assert REAL_DATA_ROOT is not None, 'Download selesai tetapi data.yaml train/val/test tidak ditemukan'
if not (OBJECT_LIBRARY / 'object_library.json').is_file():
    print('Object library belum ada; ekstraksi cutout train dimulai...')
    subprocess.run([
        sys.executable, '-u', '-m', 'coffee_detector.prepare_object_library',
        '--data-root', str(REAL_DATA_ROOT),
        '--output-root', str(OBJECT_LIBRARY),
        '--source-split', 'train',
        '--max-assets-per-class', '150',
        '--seed', '42',
    ], check=True)
print('DATASET:', REAL_DATA_ROOT)
print('CUTOUT :', OBJECT_LIBRARY)
print('OUTPUT :', OUTPUT_ROOT)

## Buat preview

Jika output dengan nama yang sama sudah ada, ubah `RUN_NAME`; generator tidak menimpa hasil lama.

In [ ]:
RUN_NAME = 'preview_seed42_v1'
RUN_ROOT = OUTPUT_ROOT / RUN_NAME
assert not RUN_ROOT.exists(), f'Output sudah ada; ubah RUN_NAME: {RUN_ROOT}'

cmd = [
    sys.executable, '-u', '-m', 'coffee_detector.run_sni_spread_preview',
    '--real-data-root', str(REAL_DATA_ROOT),
    '--object-library', str(OBJECT_LIBRARY),
    '--output-root', str(RUN_ROOT),
    '--images', '4',
    '--objects-min', '220',
    '--objects-max', '300',
    '--canvas-size', '768',
    '--seed', '42',
]
print('MULAI PREVIEW. Tunggu progres [1/4] sampai [4/4].', flush=True)
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image as DisplayImage, display

for arm in ('A1', 'A2'):
    print(f'\n{arm} — GAMBAR MENTAH')
    display(DisplayImage(filename=str(RUN_ROOT / f'{arm}_visual/contact_sheet_raw.jpg')))
    print(f'{arm} — OVERLAY ANOTASI')
    display(DisplayImage(filename=str(RUN_ROOT / f'{arm}_visual/contact_sheet.jpg')))
print('TRAINING BELUM DIJALANKAN.')